# Task 1: Import necessary libraries

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures

DATA_PATH = Path("data/largedataset_clean.txt")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/largedataset_clean.txt")


# **Task 2: Load the Dataset**

**Load your chosen dataset from the provided options and display the first few rows.**

In [ ]:
df = pd.read_csv(DATA_PATH, sep="\t")
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

# Task 3: Cleaning the Dataset
**3.1)Delete Left Hand Brain Data**

In [ ]:
# Remove left-hemisphere features; "lh_" denotes left hemisphere in these column names.
left_columns = [column for column in df.columns if column.lower().startswith("lh_") or "left" in column.lower()]
df = df.drop(columns=left_columns).copy()
df.info()


**3.2)Rounding Age**

In [ ]:
# Keep age numeric and integer-valued for the age-group analysis.
df["age"] = df["age"].round().astype(int)


**3.3)Save as a CSV File and display**


In [ ]:
df.head()


**3.4)Seperate Male and Female**

In [ ]:
# Split only after incomplete rows have been removed.


**3.5)Drop rows with missing value**

In [ ]:
df = df.dropna().copy()
female_df = df[df["gender(1=male, 2=female)"] == 2].copy()
male_df = df[df["gender(1=male, 2=female)"] == 1].copy()
print(f"Complete rows: {len(df)}; female: {len(female_df)}; male: {len(male_df)}")


**3.5)Delete Subject_label coloumn**

In [ ]:
# The source dataset spells this identifier column "subejct_label".
df = df.drop(columns=["subejct_label"], errors="ignore")
female_df = female_df.drop(columns=["subejct_label"], errors="ignore")
male_df = male_df.drop(columns=["subejct_label"], errors="ignore")


In [ ]:
print(df.columns)

# Task 4: Divide Features to Area, Thickness, Volume For Female


In [ ]:
selected_area_f = female_df.filter(regex="_area").copy()
selected_area_f.insert(0, "age", female_df["age"])
selected_thickness_f = female_df.filter(regex="thickness").copy()
selected_thickness_f.insert(0, "age", female_df["age"])
volume_f = female_df.filter(regex="Vol|volume").copy()
selected_volume_f = volume_f.copy()
selected_volume_f.insert(0, "age", female_df["age"])


# Task 5: Plot Area For Female And Show Them


In [ ]:
headers_area = selected_area_f.columns

selected_features = []

for feature_p in headers_area[1:]:
    feature_column = female_df[feature_p].values.reshape(-1, 1)
    target_column = female_df['age'].values.reshape(-1, 1)

    # Linear regression
    linear_model = LinearRegression()
    linear_model.fit(target_column, feature_column)
    linear_pred = linear_model.predict(target_column)

    # Quadratic regression
    poly_features = PolynomialFeatures(degree=2)
    target_column_poly = poly_features.fit_transform(target_column)
    quadratic_model = LinearRegression()
    quadratic_model.fit(target_column_poly, feature_column)
    quadratic_pred = quadratic_model.predict(target_column_poly)

    # Nested-model F-test (1 added quadratic term; n - 3 residual degrees of freedom)
    mse_linear = mean_squared_error(feature_column, linear_pred)
    mse_quadratic = mean_squared_error(feature_column, quadratic_pred)
    f_statistic = len(target_column) * (mse_linear - mse_quadratic) / (mse_quadratic / (len(target_column) - 3))

    # Calculate p-value
    p_value = 1 - f.cdf(f_statistic, 1, len(target_column) - 3)

    # If p-value is less than 0.05, keep the feature
    if p_value < 0.05:
        selected_features.append(feature_p)

        # Calculate RMSE for linear regression
        rmse_linear = np.sqrt(mse_linear)

        # Calculate RMSE for quadratic regression
        rmse_quadratic = np.sqrt(mse_quadratic)

        # Print RMSE values
        print(f"RMSE for {feature_p} (Linear): {rmse_linear}")
        print(f"RMSE for {feature_p} (Quadratic): {rmse_quadratic}")

        # Compare RMSE values and print the better model
        if rmse_linear < rmse_quadratic:
            print(f"Linear regression is better for {feature_p}")
        else:
            print(f"Quadratic regression is better for {feature_p}")
        

        # Calculate the turning point for quadratic regression
        quadratic_turning_point_x = -quadratic_model.coef_[0, 1] / (2 * quadratic_model.coef_[0, 2])
        quadratic_turning_point_y = quadratic_model.predict(poly_features.transform([[quadratic_turning_point_x]]))[0, 0]
        quadratic_turning_point = (round(quadratic_turning_point_x), quadratic_turning_point_y)
        print(f"Turning point for {feature_p} (Quadratic): {quadratic_turning_point}")

        # Plot the data and regressions with confidence intervals
        plt.scatter(target_column, feature_column, color='blue', label='Data')
        plt.plot(target_column, linear_pred, color='red', linewidth=3, label='Linear Regression Line')

        # Confidence interval for linear regression
        sns.lineplot(x=target_column.squeeze(), y=linear_pred.squeeze(), label='Linear Regression', color='red')

        # Plot quadratic regression curve
        quadratic_line = np.linspace(min(target_column), max(target_column), 100).reshape(-1, 1)
        quadratic_line_poly = poly_features.transform(quadratic_line)
        quadratic_line_pred = quadratic_model.predict(quadratic_line_poly)
        plt.plot(quadratic_line, quadratic_line_pred, color='green', linewidth=3, label='Quadratic Regression Curve')

        # Confidence interval for quadratic regression
        sns.lineplot(x=quadratic_line.squeeze(), y=quadratic_line_pred.squeeze(), label='Quadratic Regression', color='green')

        plt.title(f'Regression Analysis: {feature_p} vs Age:RMSE(L,Q)={rmse_linear},{rmse_quadratic}')
        plt.xlabel('Age')
        plt.ylabel(feature_p)
        plt.legend()
        plt.show()


# Print the selected features
print("Selected Features:", selected_features)


# Task 6: Plot Thickness For Female And Show Them

In [ ]:
headers_thickness = selected_thickness_f.columns

selected_features = []

for feature_p in headers_thickness[1:]:
    feature_column = female_df[feature_p].values.reshape(-1, 1)
    target_column = female_df['age'].values.reshape(-1, 1)

    # Linear regression
    linear_model = LinearRegression()
    linear_model.fit(target_column, feature_column)
    linear_pred = linear_model.predict(target_column)

    # Quadratic regression
    poly_features = PolynomialFeatures(degree=2)
    target_column_poly = poly_features.fit_transform(target_column)
    quadratic_model = LinearRegression()
    quadratic_model.fit(target_column_poly, feature_column)
    quadratic_pred = quadratic_model.predict(target_column_poly)

    # Nested-model F-test (1 added quadratic term; n - 3 residual degrees of freedom)
    mse_linear = mean_squared_error(feature_column, linear_pred)
    mse_quadratic = mean_squared_error(feature_column, quadratic_pred)
    f_statistic = len(target_column) * (mse_linear - mse_quadratic) / (mse_quadratic / (len(target_column) - 3))

    # Calculate p-value
    p_value = 1 - f.cdf(f_statistic, 1, len(target_column) - 3)

    # If p-value is less than 0.05, keep the feature
    if p_value < 0.05:
        selected_features.append(feature_p)

        # Calculate RMSE for linear regression
        rmse_linear = np.sqrt(mse_linear)

        # Calculate RMSE for quadratic regression
        rmse_quadratic = np.sqrt(mse_quadratic)

        # Print RMSE values
        print(f"RMSE for {feature_p} (Linear): {rmse_linear}")
        print(f"RMSE for {feature_p} (Quadratic): {rmse_quadratic}")

        # Compare RMSE values and print the better model
        if rmse_linear < rmse_quadratic:
            print(f"Linear regression is better for {feature_p}")
        else:
            print(f"Quadratic regression is better for {feature_p}")
        

        # Calculate the turning point for quadratic regression
        quadratic_turning_point_x = -quadratic_model.coef_[0, 1] / (2 * quadratic_model.coef_[0, 2])
        quadratic_turning_point_y = quadratic_model.predict(poly_features.transform([[quadratic_turning_point_x]]))[0, 0]
        quadratic_turning_point = (round(quadratic_turning_point_x), quadratic_turning_point_y)
        print(f"Turning point for {feature_p} (Quadratic): {quadratic_turning_point}")

        # Plot the data and regressions with confidence intervals
        plt.scatter(target_column, feature_column, color='blue', label='Data')
        plt.plot(target_column, linear_pred, color='red', linewidth=3, label='Linear Regression Line')

        # Confidence interval for linear regression
        sns.lineplot(x=target_column.squeeze(), y=linear_pred.squeeze(), label='Linear Regression', color='red')

        # Plot quadratic regression curve
        quadratic_line = np.linspace(min(target_column), max(target_column), 100).reshape(-1, 1)
        quadratic_line_poly = poly_features.transform(quadratic_line)
        quadratic_line_pred = quadratic_model.predict(quadratic_line_poly)
        plt.plot(quadratic_line, quadratic_line_pred, color='green', linewidth=3, label='Quadratic Regression Curve')

        # Confidence interval for quadratic regression
        sns.lineplot(x=quadratic_line.squeeze(), y=quadratic_line_pred.squeeze(), label='Quadratic Regression', color='green')

        plt.title(f'Regression Analysis: {feature_p} vs Age:RMSE(L,Q)={rmse_linear},{rmse_quadratic}')
        plt.xlabel('Age')
        plt.ylabel(feature_p)
        plt.legend()
        plt.show()

# Print the selected features
print("Selected Features:", selected_features)


# Task 7: Plot Volume For Female And Show Them

In [ ]:
headers_volume = selected_volume_f.columns

selected_features = []

for feature_p in headers_volume[1:]:
    feature_column = female_df[feature_p].values.reshape(-1, 1)
    target_column = female_df['age'].values.reshape(-1, 1)

    # Linear regression
    linear_model = LinearRegression()
    linear_model.fit(target_column, feature_column)
    linear_pred = linear_model.predict(target_column)

    # Quadratic regression
    poly_features = PolynomialFeatures(degree=2)
    target_column_poly = poly_features.fit_transform(target_column)
    quadratic_model = LinearRegression()
    quadratic_model.fit(target_column_poly, feature_column)
    quadratic_pred = quadratic_model.predict(target_column_poly)

    # Nested-model F-test (1 added quadratic term; n - 3 residual degrees of freedom)
    mse_linear = mean_squared_error(feature_column, linear_pred)
    mse_quadratic = mean_squared_error(feature_column, quadratic_pred)
    f_statistic = len(target_column) * (mse_linear - mse_quadratic) / (mse_quadratic / (len(target_column) - 3))

    # Calculate p-value
    p_value = 1 - f.cdf(f_statistic, 1, len(target_column) - 3)

    # If p-value is less than 0.05, keep the feature
    if p_value < 0.05:
        selected_features.append(feature_p)

        # Calculate RMSE for linear regression
        rmse_linear = np.sqrt(mse_linear)

        # Calculate RMSE for quadratic regression
        rmse_quadratic = np.sqrt(mse_quadratic)

        # Print RMSE values
        print(f"RMSE for {feature_p} (Linear): {rmse_linear}")
        print(f"RMSE for {feature_p} (Quadratic): {rmse_quadratic}")

        # Compare RMSE values and print the better model
        if rmse_linear < rmse_quadratic:
            print(f"Linear regression is better for {feature_p}")
        else:
            print(f"Quadratic regression is better for {feature_p}")
        

        # Calculate the turning point for quadratic regression
        quadratic_turning_point_x = -quadratic_model.coef_[0, 1] / (2 * quadratic_model.coef_[0, 2])
        quadratic_turning_point_y = quadratic_model.predict(poly_features.transform([[quadratic_turning_point_x]]))[0, 0]
        quadratic_turning_point = (round(quadratic_turning_point_x), quadratic_turning_point_y)
        print(f"Turning point for {feature_p} (Quadratic): {quadratic_turning_point}")

        # Plot the data and regressions with confidence intervals
        plt.scatter(target_column, feature_column, color='blue', label='Data')
        plt.plot(target_column, linear_pred, color='red', linewidth=3, label='Linear Regression Line')

        # Confidence interval for linear regression
        sns.lineplot(x=target_column.squeeze(), y=linear_pred.squeeze(), label='Linear Regression', color='red')

        # Plot quadratic regression curve
        quadratic_line = np.linspace(min(target_column), max(target_column), 100).reshape(-1, 1)
        quadratic_line_poly = poly_features.transform(quadratic_line)
        quadratic_line_pred = quadratic_model.predict(quadratic_line_poly)
        plt.plot(quadratic_line, quadratic_line_pred, color='green', linewidth=3, label='Quadratic Regression Curve')

        # Confidence interval for quadratic regression
        sns.lineplot(x=quadratic_line.squeeze(), y=quadratic_line_pred.squeeze(), label='Quadratic Regression', color='green')

        plt.title(f'Regression Analysis: {feature_p} vs Age:RMSE(L,Q)={rmse_linear},{rmse_quadratic}')
        plt.xlabel('Age')
        plt.ylabel(feature_p)
        plt.legend()
        plt.show()

# Print the selected features
print("Selected Features:", selected_features)


# Task 8: Divide Features to Area, Thickness, Volume For Male

In [ ]:
selected_area_m = male_df.filter(regex="_area").copy()
selected_area_m.insert(0, "age", male_df["age"])
selected_thickness_m = male_df.filter(regex="thickness").copy()
selected_thickness_m.insert(0, "age", male_df["age"])
volume_m = male_df.filter(regex="Vol|volume").copy()
selected_volume_m = volume_m.copy()
selected_volume_m.insert(0, "age", male_df["age"])


# Task 9: Plot Area For Male And Show Them

In [ ]:
headers_area = selected_area_m.columns

selected_features = []

for feature_p in headers_area[1:]:
    feature_column = male_df[feature_p].values.reshape(-1, 1)
    target_column = male_df['age'].values.reshape(-1, 1)

    # Linear regression
    linear_model = LinearRegression()
    linear_model.fit(target_column, feature_column)
    linear_pred = linear_model.predict(target_column)

    # Quadratic regression
    poly_features = PolynomialFeatures(degree=2)
    target_column_poly = poly_features.fit_transform(target_column)
    quadratic_model = LinearRegression()
    quadratic_model.fit(target_column_poly, feature_column)
    quadratic_pred = quadratic_model.predict(target_column_poly)

    # Nested-model F-test (1 added quadratic term; n - 3 residual degrees of freedom)
    mse_linear = mean_squared_error(feature_column, linear_pred)
    mse_quadratic = mean_squared_error(feature_column, quadratic_pred)
    f_statistic = len(target_column) * (mse_linear - mse_quadratic) / (mse_quadratic / (len(target_column) - 3))

    # Calculate p-value
    p_value = 1 - f.cdf(f_statistic, 1, len(target_column) - 3)

    # If p-value is less than 0.05, keep the feature
    if p_value < 0.05:
        selected_features.append(feature_p)

        # Calculate RMSE for linear regression
        rmse_linear = np.sqrt(mse_linear)

        # Calculate RMSE for quadratic regression
        rmse_quadratic = np.sqrt(mse_quadratic)

        # Print RMSE values
        print(f"RMSE for {feature_p} (Linear): {rmse_linear}")
        print(f"RMSE for {feature_p} (Quadratic): {rmse_quadratic}")

        # Compare RMSE values and print the better model
        if rmse_linear < rmse_quadratic:
            print(f"Linear regression is better for {feature_p}")
        else:
            print(f"Quadratic regression is better for {feature_p}")
        

        # Calculate the turning point for quadratic regression
        quadratic_turning_point_x = -quadratic_model.coef_[0, 1] / (2 * quadratic_model.coef_[0, 2])
        quadratic_turning_point_y = quadratic_model.predict(poly_features.transform([[quadratic_turning_point_x]]))[0, 0]
        quadratic_turning_point = (round(quadratic_turning_point_x), quadratic_turning_point_y)
        print(f"Turning point for {feature_p} (Quadratic): {quadratic_turning_point}")

        # Plot the data and regressions with confidence intervals
        plt.scatter(target_column, feature_column, color='blue', label='Data')
        plt.plot(target_column, linear_pred, color='red', linewidth=3, label='Linear Regression Line')

        # Confidence interval for linear regression
        sns.lineplot(x=target_column.squeeze(), y=linear_pred.squeeze(), label='Linear Regression', color='red')

        # Plot quadratic regression curve
        quadratic_line = np.linspace(min(target_column), max(target_column), 100).reshape(-1, 1)
        quadratic_line_poly = poly_features.transform(quadratic_line)
        quadratic_line_pred = quadratic_model.predict(quadratic_line_poly)
        plt.plot(quadratic_line, quadratic_line_pred, color='green', linewidth=3, label='Quadratic Regression Curve')

        # Confidence interval for quadratic regression
        sns.lineplot(x=quadratic_line.squeeze(), y=quadratic_line_pred.squeeze(), label='Quadratic Regression', color='green')

        plt.title(f'Regression Analysis: {feature_p} vs Age:RMSE(L,Q)={rmse_linear},{rmse_quadratic}')
        plt.xlabel('Age')
        plt.ylabel(feature_p)
        plt.legend()
        plt.show()

# Print the selected features
print("Selected Features:", selected_features)


# Task 10: Plot Thickness For Male And Show Them

In [ ]:
headers_thickness = selected_thickness_m.columns

selected_features = []

for feature_p in headers_thickness[1:]:
    feature_column = male_df[feature_p].values.reshape(-1, 1)
    target_column = male_df['age'].values.reshape(-1, 1)

    # Linear regression
    linear_model = LinearRegression()
    linear_model.fit(target_column, feature_column)
    linear_pred = linear_model.predict(target_column)

    # Quadratic regression
    poly_features = PolynomialFeatures(degree=2)
    target_column_poly = poly_features.fit_transform(target_column)
    quadratic_model = LinearRegression()
    quadratic_model.fit(target_column_poly, feature_column)
    quadratic_pred = quadratic_model.predict(target_column_poly)

    # Nested-model F-test (1 added quadratic term; n - 3 residual degrees of freedom)
    mse_linear = mean_squared_error(feature_column, linear_pred)
    mse_quadratic = mean_squared_error(feature_column, quadratic_pred)
    f_statistic = len(target_column) * (mse_linear - mse_quadratic) / (mse_quadratic / (len(target_column) - 3))

    # Calculate p-value
    p_value = 1 - f.cdf(f_statistic, 1, len(target_column) - 3)

    # If p-value is less than 0.05, keep the feature
    if p_value < 0.05:
        selected_features.append(feature_p)

        # Calculate RMSE for linear regression
        rmse_linear = np.sqrt(mse_linear)

        # Calculate RMSE for quadratic regression
        rmse_quadratic = np.sqrt(mse_quadratic)

        # Print RMSE values
        print(f"RMSE for {feature_p} (Linear): {rmse_linear}")
        print(f"RMSE for {feature_p} (Quadratic): {rmse_quadratic}")

        # Compare RMSE values and print the better model
        if rmse_linear < rmse_quadratic:
            print(f"Linear regression is better for {feature_p}")
        else:
            print(f"Quadratic regression is better for {feature_p}")
        

        # Calculate the turning point for quadratic regression
        quadratic_turning_point_x = -quadratic_model.coef_[0, 1] / (2 * quadratic_model.coef_[0, 2])
        quadratic_turning_point_y = quadratic_model.predict(poly_features.transform([[quadratic_turning_point_x]]))[0, 0]
        quadratic_turning_point = (round(quadratic_turning_point_x), quadratic_turning_point_y)
        print(f"Turning point for {feature_p} (Quadratic): {quadratic_turning_point}")

        # Plot the data and regressions with confidence intervals
        plt.scatter(target_column, feature_column, color='blue', label='Data')
        plt.plot(target_column, linear_pred, color='red', linewidth=3, label='Linear Regression Line')

        # Confidence interval for linear regression
        sns.lineplot(x=target_column.squeeze(), y=linear_pred.squeeze(), label='Linear Regression', color='red')

        # Plot quadratic regression curve
        quadratic_line = np.linspace(min(target_column), max(target_column), 100).reshape(-1, 1)
        quadratic_line_poly = poly_features.transform(quadratic_line)
        quadratic_line_pred = quadratic_model.predict(quadratic_line_poly)
        plt.plot(quadratic_line, quadratic_line_pred, color='green', linewidth=3, label='Quadratic Regression Curve')

        # Confidence interval for quadratic regression
        sns.lineplot(x=quadratic_line.squeeze(), y=quadratic_line_pred.squeeze(), label='Quadratic Regression', color='green')

        plt.title(f'Regression Analysis: {feature_p} vs Age:RMSE(L,Q)={rmse_linear},{rmse_quadratic}')
        plt.xlabel('Age')
        plt.ylabel(feature_p)
        plt.legend()
        plt.show()

# Print the selected features
print("Selected Features:", selected_features)


# Task 11: Plot Volume For Male And Show Them

In [ ]:
headers_volume = selected_volume_m.columns

selected_features = []

for feature_p in headers_volume[1:]:
    feature_column = male_df[feature_p].values.reshape(-1, 1)
    target_column = male_df['age'].values.reshape(-1, 1)

    # Linear regression
    linear_model = LinearRegression()
    linear_model.fit(target_column, feature_column)
    linear_pred = linear_model.predict(target_column)

    # Quadratic regression
    poly_features = PolynomialFeatures(degree=2)
    target_column_poly = poly_features.fit_transform(target_column)
    quadratic_model = LinearRegression()
    quadratic_model.fit(target_column_poly, feature_column)
    quadratic_pred = quadratic_model.predict(target_column_poly)

    # Nested-model F-test (1 added quadratic term; n - 3 residual degrees of freedom)
    mse_linear = mean_squared_error(feature_column, linear_pred)
    mse_quadratic = mean_squared_error(feature_column, quadratic_pred)
    f_statistic = len(target_column) * (mse_linear - mse_quadratic) / (mse_quadratic / (len(target_column) - 3))

    # Calculate p-value
    p_value = 1 - f.cdf(f_statistic, 1, len(target_column) - 3)

    # If p-value is less than 0.05, keep the feature
    if p_value < 0.05:
        selected_features.append(feature_p)

        # Calculate RMSE for linear regression
        rmse_linear = np.sqrt(mse_linear)

        # Calculate RMSE for quadratic regression
        rmse_quadratic = np.sqrt(mse_quadratic)

        # Print RMSE values
        print(f"RMSE for {feature_p} (Linear): {rmse_linear}")
        print(f"RMSE for {feature_p} (Quadratic): {rmse_quadratic}")

        # Compare RMSE values and print the better model
        if rmse_linear < rmse_quadratic:
            print(f"Linear regression is better for {feature_p}")
        else:
            print(f"Quadratic regression is better for {feature_p}")
        

        # Calculate the turning point for quadratic regression
        quadratic_turning_point_x = -quadratic_model.coef_[0, 1] / (2 * quadratic_model.coef_[0, 2])
        quadratic_turning_point_y = quadratic_model.predict(poly_features.transform([[quadratic_turning_point_x]]))[0, 0]
        quadratic_turning_point = (round(quadratic_turning_point_x), quadratic_turning_point_y)
        print(f"Turning point for {feature_p} (Quadratic): {quadratic_turning_point}")

        # Plot the data and regressions with confidence intervals
        plt.scatter(target_column, feature_column, color='blue', label='Data')
        plt.plot(target_column, linear_pred, color='red', linewidth=3, label='Linear Regression Line')

        # Confidence interval for linear regression
        sns.lineplot(x=target_column.squeeze(), y=linear_pred.squeeze(), label='Linear Regression', color='red')

        # Plot quadratic regression curve
        quadratic_line = np.linspace(min(target_column), max(target_column), 100).reshape(-1, 1)
        quadratic_line_poly = poly_features.transform(quadratic_line)
        quadratic_line_pred = quadratic_model.predict(quadratic_line_poly)
        plt.plot(quadratic_line, quadratic_line_pred, color='green', linewidth=3, label='Quadratic Regression Curve')

        # Confidence interval for quadratic regression
        sns.lineplot(x=quadratic_line.squeeze(), y=quadratic_line_pred.squeeze(), label='Quadratic Regression', color='green')

        plt.title(f'Regression Analysis: {feature_p} vs Age:RMSE(L,Q)={rmse_linear},{rmse_quadratic}')
        plt.xlabel('Age')
        plt.ylabel(feature_p)
        plt.legend()
        plt.show()

# Print the selected features
print("Selected Features:", selected_features)
